## 1. Load Packages

In [3]:
import re
import pandas as pd


## 2. Define chief complaint dictionary

In [4]:
complaint_dict = {"chiefcom_chest_pain" : "chest pain|tight chest|pressure chest|angina|retrosternal pain", "chiefcom_abdominal_pain" : "abdominal pain|abd pain|stomach ache|belly pain|tummy ache", 
"chiefcom_headache" : "headache|lightheaded|migraine|cephalgia|pressure head", "chiefcom_shortness_of_breath" : "breath", "chiefcom_back_pain" : "back pain|lower back pain|lumbago|upper back pain", "chiefcom_cough" : "cough|dry cough|productive cough|hacking cough", 
"chiefcom_nausea_vomiting" : "nausea|vomit|vomiting|emesis|throwing up", "chiefcom_fever_chills" : "fever|chill|temperature|feverish|shivering", "chiefcom_syncope" :"syncope|faint|fainted|blackout|loss of consciousness", "chiefcom_dizziness" : "dizz|dizzy|vertigo|lightheaded|spinning|off balance"}

## 3. Define chief complaint encoding function

In [5]:
def encode_chief_complaints(df_master, complaint_dict):
    holder_list = []
    complaint_colnames_list = list(complaint_dict.keys())
    complaint_regex_list = list(complaint_dict.values())

    for _, row in df_master.iterrows():
        curr_patient_complaint = str(row['CC'])
        curr_patient_complaint_list = [False] * len(complaint_regex_list)

        for idx, complaint in enumerate(complaint_regex_list):
            if re.search(complaint, curr_patient_complaint, re.IGNORECASE):
                curr_patient_complaint_list[idx] = True

        holder_list.append(curr_patient_complaint_list)

    df_encoded_complaint = pd.DataFrame(holder_list, columns=complaint_colnames_list)
    df_master = pd.concat([df_master, df_encoded_complaint], axis=1)
    return df_master

## 4. Read master dataset

In [6]:
df_master = pd.read_csv("/users/5/xuk/Mc_Med/visits.csv")  # Change to your true path

## 5. Implement encoding function to generate boolean label

In [7]:
df_master = encode_chief_complaints(df_master, complaint_dict)

## 6. Create statistics for each label

In [8]:
summary = (
    df_master[list(complaint_dict.keys())]
    .sum()
    .to_frame(name="count")
)
summary["proportion"] = summary["count"] / len(df_master)
summary = summary.sort_values("count", ascending=False)

print(summary)

                              count  proportion
chiefcom_abdominal_pain       15294    0.129189
chiefcom_shortness_of_breath  10255    0.086624
chiefcom_chest_pain            9776    0.082578
chiefcom_headache              4703    0.039726
chiefcom_fever_chills          4646    0.039245
chiefcom_back_pain             4038    0.034109
chiefcom_dizziness             3812    0.032200
chiefcom_nausea_vomiting       2827    0.023880
chiefcom_cough                 2548    0.021523
chiefcom_syncope                527    0.004452


## 7. Save the output

In [ ]:
df_master.to_csv("/users/5/xuk/Mc_Med/master_dataset_with_chiefcom.csv", index=False)
summary.to_csv("/users/5/xuk/Mc_Med/chiefcom_summary.csv")